<a href="https://colab.research.google.com/github/arshpreetw11/PIZZA_SUSHI_STEAK_PREDICTOR/blob/main/Milestone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Milestone Project 1: FoodVision Mini Experiment Tracking
###how do I track my machine learning experiments?

In [ ]:
try:
  import torch
  import torchvision
  assert int(torch.__version__.split(".")[1]) >= 12, "torch version should be 1.12+"
  assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
  print(f"torch version: {torch.__version__}")
  print(f"torchvision version: {torchvision.__version__}")
except:
  print(f"[INFO] torch/torchvision not as required,installing nightly versions")
  !pip install -U torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu113
  import torch
  import torchvision
  print(f"torch version: {torch.__version__}")
  print(f"torchvision version: {torchvision.__version__}")

In [ ]:
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torchvision import transforms
try:
  from torchinfo import summary
except:
  print(f"Couldn't import torchinfo, installing it...")
  !pip install -q torchinfo
  from torchinfo import summary

try:
  from going_modular.going_modular import data_setup, engine
except:
  print(f"Couldn't import going_modular, installing it...")
  #taking data from mrdbourke!!
  !git clone https://github.com/mrdbourke/pytorch-deep-learning
  !mv pytorch-deep-learning/going_modular .
  !rm -rf pytorch-deep-learning
  from going_modular.going_modular import data_setup, engine

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

##Creating a function to set Seeds..

In [ ]:
def set_seeds(seed:int=42):
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)

In [ ]:
import os
import zipfile
from pathlib import Path
import requests
import shutil # Import shutil for rmtree

def download_data(source:str,
                 destination:str,
                 remove_source:bool=True)->Path:

    data_path=Path("data/")
    image_path=data_path/destination

    # Check if the expected 'train' and 'test' subdirectories exist
    train_dir_exists = (image_path / "train").is_dir()
    test_dir_exists = (image_path / "test").is_dir()

    if image_path.is_dir() and train_dir_exists and test_dir_exists:
        print(f"[INFO] {image_path} directory already exists with 'train' and 'test' subdirectories, skipping download/unzip.")
    else:
        if image_path.is_dir(): # Directory exists but is not properly populated, clean it.
            print(f"[INFO] {image_path} directory exists but is missing 'train' or 'test' subdirectories. Clearing and re-downloading/unzipping.")
            shutil.rmtree(image_path)

        print(f"[INFO] creating directory {image_path}")
        image_path.mkdir(parents=True,exist_ok=True)

        target_file=Path(source).name
        with open(data_path/target_file,"wb") as f:
            request=requests.get(source)
            print(f"[INFO] downloading {target_file} from {source}...")
            f.write(request.content)

        with zipfile.ZipFile(data_path/target_file,"r") as zip_ref:
            print(f"[INFO] unzipping {target_file} data...")
            zip_ref.extractall(image_path)

        if remove_source:
            os.remove(data_path/target_file)
    return image_path
image_path=download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

In [ ]:
train_dir=image_path/"train"
test_dir=image_path/"test"

normalize=transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

manual_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    normalize
])

print(f"manual transform: {manual_transform}")

train_dataloader,test_dataloader,class_names=data_setup.create_dataloaders(train_dir=train_dir,
                                                                           test_dir=test_dir,
                                                                           transform=manual_transform,
                                                                           batch_size=32
)
train_dataloader,test_dataloader,class_names

Making a automatic transformer!!

In [ ]:
train_dir=image_path/"train"
test_dir=image_path/"test"

weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT
automatic_transforms=weights.transforms()
print(f"Automatic transforms: {automatic_transforms}")

train_dataloader,test_dataloader,class_names=data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=automatic_transforms,
    batch_size=32
)
train_dataloader,test_dataloader,class_names

##Getting a pretrained model

In [ ]:
weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT
models=torchvision.models.efficientnet_b0(weights=weights).to(device)

In [ ]:
#Freeze all base layers by setting up require_grad as False!!
for param in models.features.parameters():
  param.requires_grad=False

set_seeds()

models.classifier=nn.Sequential(
    nn.Dropout(p=0.2,inplace=True),
    nn.Linear(in_features=1280,
              out_features=len(class_names),
              bias=True)
)

In [ ]:
from torchinfo import summary

summary(model=models,
        input_size=(32,3,224,224),
        col_names=["input_size","output_size","num_params","trainable"],
        col_width=20,
        row_settings=["var_names"],
        device=device)

In [ ]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=models.parameters(),lr=0.001)

In [ ]:
try:
  from torch.utils.tensorboard import SummaryWriter
except:
  print(f"[INFO] Couldn't find tensorboard ,installing it ")
  !pip install -q tensorboard
  from torch.utils.tensorboard import SummaryWriter
writer=SummaryWriter()

In [ ]:
from typing import List,Dict
from tqdm.auto import tqdm
from going_modular.going_modular.engine import train_step,test_step

def train(model:torch.nn.Module,
          train_dataloader:torch.utils.data.DataLoader,
          test_dataloader:torch.utils.data.DataLoader,
          loss_fn:torch.nn.Module,
          optimizer:torch.optim.Optimizer,
          device=torch.device,
          epoch:int=5)->Dict[str,List]:

          results={
              "train_loss":[],
              "train_acc":[],
              "test_loss":[],
              "test_acc":[] # Corrected: Added 'test_acc' and removed duplicate 'test_loss'
          }
          for epoch in tqdm(range(epoch)):
            train_loss,train_acc=train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)
            test_loss,test_acc=test_step(model=model,
                                           dataloader=test_dataloader,
                                           loss_fn=loss_fn,
                                           device=device)
            print(f"Epoch: {epoch+1} | train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | test_loss: {test_loss:.4f} | test_acc: {test_acc:.4f}")
            results["train_loss"].append(train_loss)
            results["train_acc"].append(train_acc)
            results["test_loss"].append(test_loss)
            results["test_acc"].append(test_acc)

            writer.add_scalars(main_tag="Loss",
                           tag_scalar_dict={
                               "train_loss": train_loss,
                               "test_loss": test_loss
                           },
                           global_step=epoch)


            writer.add_scalars(main_tag="Accuracy",
                           tag_scalar_dict={
                               "train_acc": train_acc,
                               "test_acc": test_acc
                           },
                           global_step=epoch)
            writer.add_graph(model=model,
                             input_to_model=torch.randn(32, 3, 224, 224).to(device))
          writer.close()
          return results

In [ ]:
set_seeds()
results=train(model=models,
              train_dataloader=train_dataloader,
              test_dataloader=test_dataloader,
              loss_fn=loss_fn,
              optimizer=optimizer,
              device=device,
              epoch=3)

In [ ]:
results

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
def create_writer(experiment_name:str,
                  model_name:str,
                  extra:str=None)->torch.utils.tensorboard.writer.SummaryWriter():
                  from datetime import datetime
                  import os

                  timestamp=datetime.now().strftime("%Y-%m-%d")
                  if extra:
                    log_dir=os.path.join("runs",experiment_name,model_name,timestamp,extra)
                  else:
                    log_dir=os.path.join("runs",experiment_name,model_name,timestamp)
                  print(f"[INFO] saving to {log_dir}")
                  return SummaryWriter(log_dir=log_dir)



In [ ]:
example_writer = create_writer(experiment_name="data_10_percent",
                               model_name="effnetb0",
                               extra="5_epochs")

In [ ]:
from typing import List,Dict
from tqdm.auto import tqdm
from going_modular.going_modular.engine import train_step,test_step

def train(model:torch.nn.Module,
          train_dataloader:torch.utils.data.DataLoader,
          test_dataloader:torch.utils.data.DataLoader,
          loss_fn:torch.nn.Module,
          optimizer:torch.optim.Optimizer,
          device:torch.device,
          epoch:int,
          writer:torch.utils.tensorboard.writer.SummaryWriter)->Dict[str,List]:

          results={
              "train_loss":[],
              "train_acc":[],
              "test_loss":[],
              "test_acc":[]
          }
          for epoch in tqdm(range(epoch)):
            train_loss,train_acc=train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)
            test_loss,test_acc=test_step(model=model,
                                           dataloader=test_dataloader,
                                           loss_fn=loss_fn,
                                           device=device)
            print(f"Epoch: {epoch+1} | train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | test_loss: {test_loss:.4f} | test_acc: {test_acc:.4f}")
            results["train_loss"].append(train_loss)
            results["train_acc"].append(train_acc)
            results["test_loss"].append(test_loss)
            results["test_acc"].append(test_acc)

            if writer:

                writer.add_scalars(main_tag="Loss",
                              tag_scalar_dict={
                                  "train_loss": train_loss,
                                  "test_loss": test_loss
                              },
                              global_step=epoch)


                writer.add_scalars(main_tag="Accuracy",
                              tag_scalar_dict={
                                  "train_acc": train_acc,
                                  "test_acc": test_acc
                              },
                              global_step=epoch)
                writer.close()
            else:

                pass


          return results

In [ ]:
data_10_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                                     destination="pizza_steak_sushi")

data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

In [ ]:
train_dir_10_percent=data_10_percent_path/"train"
train_dir_20_percent=data_20_percent_path/"train"

test_dir=data_10_percent_path/"test"
print(f"Training directory 10%: {train_dir_10_percent}")
print(f"Training directory 20%: {train_dir_20_percent}")
print(f"Testing directory: {test_dir}")

In [ ]:
from torchvision import transforms
normalize=transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    normalize
])

BATCH_SIZE=32

train_dataloader_10_percent,test_dataloader_10_percent,class_names=data_setup.create_dataloaders(train_dir=train_dir_10_percent,
                                                                                              test_dir=test_dir,
                                                                                              transform=transform,
                                                                                              batch_size=BATCH_SIZE)
train_dataloader_20_percent,test_dataloader_20_percent,class_names=data_setup.create_dataloaders(train_dir=train_dir_20_percent,
                                                                                              test_dir=test_dir,
                                                                                              transform=transform,
                                                                                              batch_size=BATCH_SIZE)
print(f"Number of batches of size {BATCH_SIZE} in 10 percent training data: {len(train_dataloader_10_percent)}")
print(f"Number of batches of size {BATCH_SIZE} in 20 percent training data: {len(train_dataloader_20_percent)}")
print(f"Number of batches of size {BATCH_SIZE} in testing data: {len(test_dataloader)} (all experiments will use the same test set)")
print(f"Number of classes: {len(class_names)}, class names: {class_names}")

In [ ]:
import torchvision
from torchinfo import summary

effnetb2_weights=torchvision.models.EfficientNet_B2_Weights.DEFAULT
effnetb2_transforms=effnetb2_weights.transforms()
effnetb2_model=torchvision.models.efficientnet_b2(weights=effnetb2_weights)

In [ ]:
print(f"Number of in_features to final layer of EfficientNetB2: {len(effnetb2_model.classifier.state_dict()['1.weight'][0])}")

In [ ]:
summary(model=effnetb2_model,
        input_size=(32,3,224,224),
        col_names=["input_size","output_size","num_params","trainable"],
        col_width=20,
        row_settings=["var_names"],
        device=device)

In [ ]:
import torchvision
from torch import nn

OUT_FEATURE=len(class_names)

def create_effnetb0():
  weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT
  models=torchvision.models.efficientnet_b0(weights=weights)

  for params in models.features.parameters():
    params.requires_grad=False

  set_seeds()

  models.classifier=nn.Sequential(
      nn.Dropout(p=0.2),
      nn.Linear(in_features=1280,
                out_features=OUT_FEATURE,
                bias=True)
  )

  model_name="effnetb0"
  print(f"[INFO] created new {model_name} model.")
  return models

def create_effnetb2():
  weights=torchvision.models.EfficientNet_B2_Weights.DEFAULT
  models=torchvision.models.efficientnet_b2(weights=weights)

  for params in models.features.parameters():
    params.requires_grad=False

  set_seeds()

  models.classifier=nn.Sequential(
      nn.Dropout(p=0.2),
      nn.Linear(in_features=1408,
                out_features=OUT_FEATURE,
                bias=True)
  )

  model_name="effnetb2"
  print(f"[INFO] created new {model_name} model.")
  return models


In [ ]:
effnetb0=create_effnetb0()
effnetb0

In [ ]:
effnetb2=create_effnetb2()
effnetb2


In [ ]:
num_epochs=[5,10]
models=["effnetb0","effnetb2"]
experiment_train_dataloaders={
    "data_10_percent":train_dataloader_10_percent,
    "data_20_percent":train_dataloader_20_percent
}
test_dataloader=test_dataloader
loss_fn=nn.CrossEntropyLoss()

In [ ]:
from going_modular.going_modular.utils import save_model

set_seeds()

experiment_number=0

for dataloader_name, current_train_dataloader in experiment_train_dataloaders.items():
  for epoch in num_epochs:
    for model_name in models:

      experiment_number += 1
      print(f"[INFO] Experiment number: {experiment_number}")
      print(f"[INFO] Model: {model_name}")
      print(f"[INFO] DataLoader: {dataloader_name}")
      print(f"[INFO] Number of epochs: {epoch}")

      if model_name=="effnetb0":
        model=create_effnetb0 ()
      else:
        model=create_effnetb2 ()
      model = model.to(device)
      loss_fn=torch.nn.CrossEntropyLoss()
      optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
      train(model=model,
            train_dataloader=current_train_dataloader,
            test_dataloader=test_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device,
            epoch=epoch,
            writer=create_writer(experiment_name=dataloader_name,
                                 model_name=model_name,
                                 extra=f"epochs={epoch}"))
      save_filepath=f"{model_name}_{dataloader_name}_{epoch}_epochs.pth"
      save_model(model=model,
                 target_dir="models",
                       model_name=save_filepath)
      print("-"*50 + "\n")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
best_model_path="models/effnetb2_data_20_percent_10_epochs.pth"
best_model=create_effnetb2()
best_model.to(device)
best_model.load_state_dict(torch.load(best_model_path))

In [ ]:
from pathlib import Path
effnetb2_model_size=Path(best_model_path).stat().st_size//(1024*1024)
print(f"EfficientNetB2 feature extractor model size: {effnetb2_model_size} MB")

In [ ]:
from going_modular.going_modular.predictions import pred_and_plot_image

import random
num_images_to_plot=3
test_image_path_list = list(Path(data_20_percent_path / "test").glob("*/*.jpg"))
test_image_path_sample = random.sample(population=test_image_path_list,
                                       k=num_images_to_plot)
for image_path in test_image_path_sample:
  pred_and_plot_image(model=best_model,
                      image_path=image_path,
                      class_names=class_names,
                      image_size=(224,224))

In [ ]:
import requests
custom_image_path=Path("data/04-pizza-dad.jpeg")

if not custom_image_apth.is_file():
  with open(custom_image_path, "wb") as f:
        # When downloading from GitHub, need to use the "raw" file link
        request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg")
        print(f"Downloading {custom_image_path}...")
        f.write(request.content)
else:
  print(f"{custom_image_path} already exists.")
pred_and_plot_image(model=model,
                    image_path=custom_image_path,
                    class_names=class_names)